# Fake Job Posting Fraud Detection — Clean Sequential Workflow

This notebook is a **cleaned and reordered version** of the original project notebook. It is designed to be run **top-to-bottom from a fresh kernel**.

## What was corrected during cleanup

- Removed the Google Drive dependency for loading the CSV; the notebook now searches common local/Colab paths.
- Removed duplicated EDA, duplicated text-length creation, duplicated `combined_text` creation, interrupted GridSearchCV cells, and old save/reload detours.
- Removed the old commented-out description-only LSTM experiments from the main workflow; the completed **combined-text BiLSTM** experiment is retained.
- Created the train/test row split **once** and reuse the same rows for structured, TF-IDF, and deep-learning models.
- Renamed structured/text variables so `X_train`, `X_test`, `tfidf`, and model objects are not accidentally overwritten.
- Removed all hard-coded metric tables. Every metric in the final table is generated from the model predictions produced in the current run.
- Corrected model naming: the old notebook mixed **structured Logistic Regression metrics** with the label `TF-IDF + LR`, and mixed **structured RBF-SVM metrics** with the label `TF-IDF + SVM`.
- TF-IDF is now built once and reused by LR, RF, and LinearSVC so those three text classifiers are compared on the same representation.
- SHAP is attached to the exact fitted TF-IDF Logistic Regression model and matching TF-IDF matrices.
- Plot saving occurs before `plt.show()`.

## Intentionally NOT added yet

This cleanup does **not** add the later improvements we discussed: duplicate/group-aware splitting, PR-AUC, threshold tuning, structured+text fusion, LIME, DistilBERT, FastAPI/Streamlit deployment, or other new modeling ideas. Those can be added after reviewing this notebook sequentially.

## 1. Setup

Run this section first. If a package is missing in your local environment, install it once using the commented `%pip` line.

In [ ]:
# Uncomment only if your environment is missing packages.
%pip install -q tensorflow

from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)
%matplotlib inline

## 2. Load the dataset

The original notebook hard-coded `/content/fake_job_postings.csv`. This version first checks the current folder, then common Colab/sandbox locations.

In [ ]:
DATA_CANDIDATES = [
    Path("fake_job_postings.csv"),
    Path("/content/fake_job_postings.csv"),
    Path("/mnt/data/fake_job_postings.csv"),
]

csv_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "fake_job_postings.csv was not found. Put it in the same folder as this notebook "
        "or update DATA_CANDIDATES with the correct path."
    )

raw_df = pd.read_csv(csv_path)
print("Loaded:", csv_path)
print("Shape:", raw_df.shape)
raw_df.head()

## 3. Initial data audit

Start with shape, columns, missing values, and class balance before changing the data.

In [ ]:
print("Shape:", raw_df.shape)
print()
print("Columns:")
print(raw_df.columns.tolist())

print()
print("Missing values:")
display(raw_df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print()
print("Target counts:")
print(raw_df["fraudulent"].value_counts())

print()
print("Target proportions:")
print(raw_df["fraudulent"].value_counts(normalize=True))

In [ ]:
# Missingness of selected fields by target class, preserved from the original analysis.
for col in ["salary_range", "department", "location"]:
    print()
    print(f"Missing fraction for {col} by class:")
    print(raw_df.groupby("fraudulent")[col].apply(lambda x: x.isna().mean()))

## 4. Basic cleaning and feature preparation

The original project dropped `salary_range`, filled missing `location`, and later dropped `department`. That logic is retained here. Missing values in other structured categorical columns are left as missing-category values for `OneHotEncoder(handle_unknown='ignore')`.

In [ ]:
df = raw_df.copy()

# Very high missingness in the original notebook -> dropped.
df = df.drop(columns=["salary_range"])

# Preserve the original location handling.
df["location"] = df["location"].fillna("Unknown_location")

# Department was explored but not used in the final structured model.
df = df.drop(columns=["department"])

# Text-length features used by the structured model.
TEXT_COLUMNS = ["title", "description", "requirements"]
for col in TEXT_COLUMNS:
    df[f"{col}_len"] = df[col].fillna("").str.len()

# Combined text used by TF-IDF and the completed BiLSTM experiments.
df["combined_text"] = (
    df["title"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["requirements"].fillna("")
).str.strip()

print("Cleaned shape:", df.shape)
df[["title", "description_len", "requirements_len", "combined_text", "fraudulent"]].head()

## 5. Exploratory Data Analysis

These are the useful EDA components already present in the original notebook, organized before modeling.

In [ ]:
# Categorical distributions and fraud rates.
eda_categorical = [
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function",
]

for col in eda_categorical:
    print()
    print(f"==== {col} ====")
    print(df[col].value_counts(dropna=False).head(10))
    print()
    print("Top fraud rates:")
    print(
        df.groupby(col, dropna=False)["fraudulent"]
          .mean()
          .sort_values(ascending=False)
          .head(10)
    )

In [ ]:
for col in ["telecommuting", "has_company_logo", "has_questions"]:
    print()
    print(f"==== {col} ====")
    print(df[col].value_counts())
    print("Fraud rate:")
    print(df.groupby(col)["fraudulent"].mean())

In [ ]:
for col in ["title_len", "description_len", "requirements_len"]:
    print()
    print(f"{col} statistics by class:")
    display(df.groupby("fraudulent")[col].describe())

In [ ]:
# Class distribution
class_counts = df["fraudulent"].value_counts().sort_index()
plt.figure(figsize=(4, 4))
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.title("Class distribution: Real vs Fake jobs")
plt.xlabel("Fraudulent (0 = Real, 1 = Fake)")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("class_distribution_fraudulent.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Fraud rate vs three important binary metadata fields.
for col, title, xlabel in [
    ("has_company_logo", "Fraud rate vs Company Logo", "Has company logo (0 = No, 1 = Yes)"),
    ("has_questions", "Fraud rate vs Screening Questions", "Has screening questions (0 = No, 1 = Yes)"),
    ("telecommuting", "Fraud rate vs Telecommuting", "Telecommuting (0 = No, 1 = Yes)"),
]:
    rate = df.groupby(col)["fraudulent"].mean().sort_index()
    plt.figure(figsize=(4, 4))
    plt.bar(rate.index.astype(str), rate.values)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Mean fraudulent rate")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(f"fraud_rate_vs_{col}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# Description-length distribution by class.
plt.figure(figsize=(6, 4))
for fraud_value, label in [(0, "Real"), (1, "Fake")]:
    values = df.loc[df["fraudulent"] == fraud_value, "description_len"]
    plt.hist(values, bins=60, density=True, alpha=0.4, label=label)
plt.title("Description length distribution by class")
plt.xlabel("Description length")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig("description_len_by_class_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Create one common train/test row split

This is still the original **stratified random 80/20 split** with `random_state=42`. The only cleanup is that the row indices are created once, so every model evaluates on the same rows.

> **Not added yet:** duplicate/group-aware splitting. That is a later correction we already identified and will implement separately.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Use the complete text of the job posting as the grouping key.
# Identical postings will therefore stay in the same split.
groups = df["combined_text"].fillna("")

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Take the first fold as the held-out test set.
train_idx, test_idx = next(
    sgkf.split(
        df,
        df["fraudulent"],
        groups=groups
    )
)

y_train = df.iloc[train_idx]["fraudulent"].astype(int).to_numpy()
y_test = df.iloc[test_idx]["fraudulent"].astype(int).to_numpy()

print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))

print("\nTraining fraud rate:", y_train.mean())
print("Test fraud rate:", y_test.mean())

In [ ]:
train_text = set(df.iloc[train_idx]["combined_text"])
test_text = set(df.iloc[test_idx]["combined_text"])

overlap = train_text.intersection(test_text)

print("Unique training texts:", len(train_text))
print("Unique testing texts:", len(test_text))
print("Exact text overlap:", len(overlap))

In [ ]:
print("Training class distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest class distribution:")
print(pd.Series(y_test).value_counts())

print("\nTraining fraud rate:")
print(pd.Series(y_train).mean())

print("\nTest fraud rate:")
print(pd.Series(y_test).mean())

## 7. Evaluation helper

The old notebook manually typed model metrics into later tables, which caused the TF-IDF LR/SVM labeling errors. This helper records metrics directly from the predictions made in the current run.

In [ ]:
model_results = {}

def evaluate_and_store(name, y_true, y_pred, y_score=None):
    print()
    print(name)
    print("=" * len(name))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print()
    print("Classification report:")
    print(classification_report(y_true, y_pred, digits=4))

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Fraud_precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Fraud_recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Fraud_F1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "ROC_AUC": np.nan,
    }

    if y_score is not None:
        row["ROC_AUC"] = roc_auc_score(y_true, y_score)
        print(f"ROC-AUC: {row['ROC_AUC']:.4f}")

    model_results[name] = row
    return row

# Part A — Structured-feature models

The structured models use the same fields as the original notebook: three binary metadata fields, categorical metadata, location, and text-length features.

In [ ]:
NUM_COLS = ["title_len", "description_len", "requirements_len"]
CAT_COLS = [
    "telecommuting",
    "has_company_logo",
    "has_questions",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function",
    "location",
]
STRUCTURED_FEATURES = CAT_COLS + NUM_COLS

X_struct_train = df.loc[train_idx, STRUCTURED_FEATURES]
X_struct_test = df.loc[test_idx, STRUCTURED_FEATURES]

structured_preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
        ("num", "passthrough", NUM_COLS),
    ]
)

## 8. Structured Logistic Regression baseline

This is kept as a baseline, not as the final text model. `liblinear` is used here to avoid the repeated convergence-warning detour present in the old notebook.

In [ ]:
structured_lr = Pipeline([
    ("preprocess", structured_preprocess),
    ("model", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE,
    )),
])

structured_lr.fit(X_struct_train, y_train)
structured_lr_pred = structured_lr.predict(X_struct_test)
structured_lr_score = structured_lr.predict_proba(X_struct_test)[:, 1]

evaluate_and_store(
    "Logistic Regression (structured)",
    y_test,
    structured_lr_pred,
    structured_lr_score,
)

## 9. Structured Random Forest

This is the structured model used for the feature-importance plot.

In [ ]:
structured_rf = Pipeline([
    ("preprocess", structured_preprocess),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

structured_rf.fit(X_struct_train, y_train)
structured_rf_pred = structured_rf.predict(X_struct_test)
structured_rf_score = structured_rf.predict_proba(X_struct_test)[:, 1]

evaluate_and_store(
    "Random Forest (structured)",
    y_test,
    structured_rf_pred,
    structured_rf_score,
)

## 10. Structured RBF-SVM baseline

This was present in the original notebook. It is explicitly labeled **structured** here so it cannot be confused with the later TF-IDF + LinearSVC model. `decision_function` is sufficient for ROC-AUC, so `probability=True` is not required.

In [ ]:
structured_svm = Pipeline([
    ("preprocess", structured_preprocess),
    ("model", SVC(
        kernel="rbf",
        C=1.0,
        class_weight="balanced",
        probability=False,
        random_state=RANDOM_STATE,
    )),
])

structured_svm.fit(X_struct_train, y_train)
structured_svm_pred = structured_svm.predict(X_struct_test)
structured_svm_score = structured_svm.decision_function(X_struct_test)

evaluate_and_store(
    "RBF SVM (structured)",
    y_test,
    structured_svm_pred,
    structured_svm_score,
)

## 11. Structured Random Forest feature importance

This is the cleaned version of the existing top-15 structured-feature plot. It uses the already-fitted `structured_rf` object rather than retraining it again just for the plot.

In [ ]:
rf_model = structured_rf.named_steps["model"]
ohe = structured_rf.named_steps["preprocess"].named_transformers_["cat"]

cat_feature_names = ohe.get_feature_names_out(CAT_COLS)
structured_feature_names = np.concatenate([cat_feature_names, NUM_COLS])

importances = rf_model.feature_importances_
top_n = 15
top_idx = np.argsort(importances)[::-1][:top_n]

importance_df = pd.DataFrame({
    "Feature": structured_feature_names[top_idx],
    "Importance": importances[top_idx],
}).sort_values("Importance", ascending=True)

display(importance_df.sort_values("Importance", ascending=False))

plt.figure(figsize=(8, 6))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.xlabel("Feature importance")
plt.ylabel("Feature")
plt.title("Random Forest – Top 15 Structured Feature Importances")
plt.tight_layout()
plt.savefig("rf_struct_feature_importance_top15.png", dpi=300, bbox_inches="tight")
plt.show()

# Part B — TF-IDF text models

The old notebook recreated and overwrote `tfidf`, `X_train_tfidf`, and text splits in multiple places. Here we create **one TF-IDF representation once** and use it for Logistic Regression, Random Forest, and LinearSVC.

In [ ]:
X_text_train = df.loc[train_idx, "combined_text"].fillna("")
X_text_test = df.loc[test_idx, "combined_text"].fillna("")

text_tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9,
)

X_text_train_tfidf = text_tfidf.fit_transform(X_text_train)
X_text_test_tfidf = text_tfidf.transform(X_text_test)

print("TF-IDF training matrix:", X_text_train_tfidf.shape)
print("TF-IDF test matrix:", X_text_test_tfidf.shape)

## 12. TF-IDF + Logistic Regression

This is the interpretable linear text model used later for coefficient analysis and SHAP.

In [ ]:
tfidf_lr = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    solver="liblinear",
    random_state=RANDOM_STATE,
)

tfidf_lr.fit(X_text_train_tfidf, y_train)
tfidf_lr_pred = tfidf_lr.predict(X_text_test_tfidf)
tfidf_lr_score = tfidf_lr.predict_proba(X_text_test_tfidf)[:, 1]

evaluate_and_store(
    "TF-IDF + Logistic Regression",
    y_test,
    tfidf_lr_pred,
    tfidf_lr_score,
)

## 13. TF-IDF + Random Forest

In [ ]:
tfidf_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

tfidf_rf.fit(X_text_train_tfidf, y_train)
tfidf_rf_pred = tfidf_rf.predict(X_text_test_tfidf)
tfidf_rf_score = tfidf_rf.predict_proba(X_text_test_tfidf)[:, 1]

evaluate_and_store(
    "TF-IDF + Random Forest",
    y_test,
    tfidf_rf_pred,
    tfidf_rf_score,
)

## 14. TF-IDF + LinearSVC

This is the actual **text SVM** experiment. In the old final chart, the much poorer structured RBF-SVM score was accidentally shown under the `TF-IDF + SVM` label.

In [ ]:
tfidf_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

tfidf_svm.fit(X_text_train_tfidf, y_train)
tfidf_svm_pred = tfidf_svm.predict(X_text_test_tfidf)
tfidf_svm_score = tfidf_svm.decision_function(X_text_test_tfidf)

evaluate_and_store(
    "TF-IDF + LinearSVC",
    y_test,
    tfidf_svm_pred,
    tfidf_svm_score,
)

## 15. Interpreting TF-IDF Logistic Regression coefficients

Positive coefficients push toward `fraudulent = 1`; negative coefficients push toward `fraudulent = 0`.

In [ ]:
tfidf_feature_names = np.array(text_tfidf.get_feature_names_out())
coefs = tfidf_lr.coef_[0]

top_k = 20
top_fake_idx = np.argsort(coefs)[-top_k:][::-1]
top_real_idx = np.argsort(coefs)[:top_k]

coefficient_df = pd.DataFrame({
    "Fake_words": tfidf_feature_names[top_fake_idx],
    "Fake_coef": np.round(coefs[top_fake_idx], 3),
    "Real_words": tfidf_feature_names[top_real_idx],
    "Real_coef": np.round(coefs[top_real_idx], 3),
})

coefficient_df

# Part C — Deep learning models

The original notebook contained several commented description-only LSTM attempts plus a completed combined-text BiLSTM. The main workflow below keeps only the completed **combined-text** direction so the notebook stays sequential and reproducible.

> Training time depends strongly on whether you run on CPU or GPU.

In [ ]:
%pip install -q tensorflow  # Uncomment if needed; tensorflow should already be installed from cell 2

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.utils.class_weight import compute_class_weight

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

LSTM_MAX_WORDS = 20000
LSTM_MAX_LEN = 400
LSTM_BATCH_SIZE = 128
LSTM_EPOCHS = 5

lstm_tokenizer = Tokenizer(num_words=LSTM_MAX_WORDS, oov_token="<OOV>")
lstm_tokenizer.fit_on_texts(X_text_train)

lstm_train_seq = lstm_tokenizer.texts_to_sequences(X_text_train)
lstm_test_seq = lstm_tokenizer.texts_to_sequences(X_text_test)

lstm_train_pad = pad_sequences(
    lstm_train_seq,
    maxlen=LSTM_MAX_LEN,
    padding="post",
    truncating="post",
)
lstm_test_pad = pad_sequences(
    lstm_test_seq,
    maxlen=LSTM_MAX_LEN,
    padding="post",
    truncating="post",
)

classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)
class_weights = dict(zip(classes, class_weights_array))

print("Class weights:", class_weights)
print("Padded training shape:", lstm_train_pad.shape)

## 16. Combined-text BiLSTM with randomly initialized embeddings

In [ ]:
random_lstm = models.Sequential([
    layers.Input(shape=(LSTM_MAX_LEN,)),
    layers.Embedding(input_dim=LSTM_MAX_WORDS, output_dim=128),
    layers.Bidirectional(layers.LSTM(64, return_sequences=False)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid"),
])

random_lstm.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

random_lstm.summary()

random_lstm_history = random_lstm.fit(
    lstm_train_pad,
    y_train,
    epochs=LSTM_EPOCHS,
    batch_size=LSTM_BATCH_SIZE,
    validation_split=0.2,
    class_weight=class_weights,
    verbose=1,
)

In [ ]:
random_lstm_score = random_lstm.predict(lstm_test_pad, verbose=0).ravel()
random_lstm_pred = (random_lstm_score >= 0.5).astype(int)

evaluate_and_store(
    "BiLSTM (combined_text, random embeddings)",
    y_test,
    random_lstm_pred,
    random_lstm_score,
)

## 17. GloVe embeddings + BiLSTM

This reproduces the existing GloVe experiment using `glove.6B.100d`. The file is large; the download occurs only if it is not already cached. `trainable=True` preserves the later fine-tuned version of your experiment.

In [ ]:
import zipfile

GLOVE_CACHE_DIR = Path("glove_data")
GLOVE_CACHE_DIR.mkdir(exist_ok=True)
GLOVE_TXT = GLOVE_CACHE_DIR / "glove.6B.100d.txt"

if not GLOVE_TXT.exists():
    glove_zip = tf.keras.utils.get_file(
        fname="glove.6B.zip",
        origin="https://nlp.stanford.edu/data/glove.6B.zip",
        cache_dir=str(GLOVE_CACHE_DIR.resolve()),
        cache_subdir=".",
    )
    with zipfile.ZipFile(glove_zip, "r") as zf:
        zf.extract("glove.6B.100d.txt", path=GLOVE_CACHE_DIR)

print("GloVe file:", GLOVE_TXT.resolve())

In [ ]:
embeddings_index = {}
with open(GLOVE_TXT, encoding="utf8") as f:
    for line in f:
        values = line.rstrip().split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

GLOVE_DIM = 100
word_index = lstm_tokenizer.word_index
num_words = min(LSTM_MAX_WORDS, len(word_index) + 1)

embedding_matrix = np.zeros((num_words, GLOVE_DIM), dtype="float32")
for word, i in word_index.items():
    if i >= num_words:
        continue
    vector = embeddings_index.get(word)
    if vector is not None:
        embedding_matrix[i] = vector

print("Vocabulary used:", num_words)
print("Embedding matrix shape:", embedding_matrix.shape)

In [ ]:
glove_lstm = models.Sequential([
    layers.Input(shape=(LSTM_MAX_LEN,)),
    layers.Embedding(
        input_dim=num_words,
        output_dim=GLOVE_DIM,
        weights=[embedding_matrix],
        trainable=True,
    ),
    layers.Bidirectional(layers.LSTM(64, return_sequences=False)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid"),
])

glove_lstm.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

glove_lstm.summary()

glove_lstm_history = glove_lstm.fit(
    lstm_train_pad,
    y_train,
    epochs=5,
    batch_size=LSTM_BATCH_SIZE,
    validation_split=0.2,
    class_weight=class_weights,
    verbose=1,
)

In [ ]:
glove_lstm_score = glove_lstm.predict(lstm_test_pad, verbose=0).ravel()
glove_lstm_pred = (glove_lstm_score >= 0.5).astype(int)

evaluate_and_store(
    "BiLSTM (combined_text, GloVe)",
    y_test,
    glove_lstm_pred,
    glove_lstm_score,
)

# Part D — Final comparison and explainability

The comparison table below is generated from `model_results`. There are **no manually typed F1/ROC-AUC values**, which prevents the metric-label mismatch that occurred in the old notebook.

In [ ]:
results_df = (
    pd.DataFrame(model_results.values())
      .sort_values("Fraud_F1", ascending=False)
      .reset_index(drop=True)
)

results_df.to_csv("model_metrics_from_current_run.csv", index=False)
results_df

In [ ]:
# Main-model fraud F1 plot. Structured LR/RBF-SVM remain baselines and are not shown here.
main_model_order = [
    "Random Forest (structured)",
    "TF-IDF + Logistic Regression",
    "TF-IDF + Random Forest",
    "TF-IDF + LinearSVC",
    "BiLSTM (combined_text, random embeddings)",
    "BiLSTM (combined_text, GloVe)",
]

main_f1_df = (
    results_df[results_df["Model"].isin(main_model_order)]
    .set_index("Model")
    .reindex(main_model_order)
    .reset_index()
)

plt.figure(figsize=(9, 5))
plt.bar(main_f1_df["Model"], main_f1_df["Fraud_F1"])
plt.ylim(0, 1)
plt.ylabel("Fraud F1 score")
plt.xlabel("")
plt.title("Fraud F1 for Main Models")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("fraud_f1_main_models.png", dpi=300, bbox_inches="tight")
plt.show()

## 18. SHAP for TF-IDF + Logistic Regression

This keeps the SHAP work already present in the project. It intentionally does **not** add LIME or new local-explanation UI work yet.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("shap") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])

import shap

# Use a small background sample for speed.
X_shap_background = shap.sample(X_text_train_tfidf, 200, random_state=RANDOM_STATE)
shap_masker = shap.maskers.Independent(X_shap_background)
shap_explainer = shap.LinearExplainer(tfidf_lr, shap_masker)

# Explain a small subset of the exact matching test TF-IDF matrix.
X_shap_test = X_text_test_tfidf[:50]
shap_values = shap_explainer(X_shap_test)

shap.summary_plot(
    shap_values.values,
    X_shap_test,
    feature_names=text_tfidf.get_feature_names_out(),
)

# End of cleaned baseline notebook

At this point the original project workflow is in a consistent order:

1. data audit and cleaning  
2. EDA  
3. one common stratified row split  
4. structured baselines  
5. TF-IDF text models  
6. combined-text BiLSTM  
7. GloVe + BiLSTM  
8. dynamically generated comparison table  
9. structured RF importance, LR coefficients, and SHAP  

## Corrections to do later (not implemented here)

After you have reviewed and rerun this baseline sequentially, the next revision can add the changes we already identified: duplicate/group-aware validation, PR-AUC and threshold analysis, structured+text fusion, better local SHAP + LIME, DistilBERT, and deployment.

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve
import matplotlib.pyplot as plt

# Example: TF-IDF + Logistic Regression
y_prob_lr = tfidf_lr.predict_proba(X_text_test_tfidf)[:, 1]

pr_auc_lr = average_precision_score(y_test, y_prob_lr)

print("TF-IDF + Logistic Regression")
print("PR-AUC:", round(pr_auc_lr, 4))

In [ ]:
y_score_svc = tfidf_svm.decision_function(X_text_test_tfidf)

pr_auc_svc = average_precision_score(y_test, y_score_svc)

print("TF-IDF + LinearSVC")
print("PR-AUC:", round(pr_auc_svc, 4))

In [ ]:
precision, recall, thresholds = precision_recall_curve(
    y_test,
    y_prob_lr
)

plt.figure(figsize=(7, 5))
plt.plot(recall, precision)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — TF-IDF + Logistic Regression")
plt.grid(True)
plt.show()

In [ ]:
# -------------------------------------------------------
# GROUP-AWARE TRAIN / VALIDATION / TEST SPLIT
# -------------------------------------------------------

groups = df["combined_text"].fillna("")

# First split:
# 80% development data
# 20% final test data

sgkf_test = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

dev_idx, test_idx = next(
    sgkf_test.split(
        df,
        df["fraudulent"],
        groups=groups
    )
)

# Second split:
# 80% of development -> train
# 20% of development -> validation
#
# Overall approximately:
# Train      = 64%
# Validation = 16%
# Test       = 20%

dev_df = df.iloc[dev_idx].copy()

sgkf_val = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

train_rel_idx, val_rel_idx = next(
    sgkf_val.split(
        dev_df,
        dev_df["fraudulent"],
        groups=dev_df["combined_text"]
    )
)

train_idx = dev_df.index[train_rel_idx]
val_idx = dev_df.index[val_rel_idx]

print("Train size:", len(train_idx))
print("Validation size:", len(val_idx))
print("Test size:", len(test_idx))

print("\nTrain percentage:",
      round(len(train_idx) / len(df) * 100, 2), "%")

print("Validation percentage:",
      round(len(val_idx) / len(df) * 100, 2), "%")

print("Test percentage:",
      round(len(test_idx) / len(df) * 100, 2), "%")

In [ ]:
y_train = df.loc[train_idx, "fraudulent"].astype(int)
y_val = df.loc[val_idx, "fraudulent"].astype(int)
y_test = df.loc[test_idx, "fraudulent"].astype(int)

print("Fraud rates")
print("-" * 40)

print(f"Overall:     {df['fraudulent'].mean():.4f}")
print(f"Train:       {y_train.mean():.4f}")
print(f"Validation:  {y_val.mean():.4f}")
print(f"Test:        {y_test.mean():.4f}")

In [ ]:
train_texts = set(df.loc[train_idx, "combined_text"])
val_texts = set(df.loc[val_idx, "combined_text"])
test_texts = set(df.loc[test_idx, "combined_text"])

train_val_overlap = train_texts.intersection(val_texts)
train_test_overlap = train_texts.intersection(test_texts)
val_test_overlap = val_texts.intersection(test_texts)

print("Train ↔ Validation overlap:", len(train_val_overlap))
print("Train ↔ Test overlap:", len(train_test_overlap))
print("Validation ↔ Test overlap:", len(val_test_overlap))

In [ ]:
X_train_text = df.loc[train_idx, "combined_text"].fillna("")
X_val_text = df.loc[val_idx, "combined_text"].fillna("")
X_test_text = df.loc[test_idx, "combined_text"].fillna("")

In [ ]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train_text)

X_val_tfidf = tfidf.transform(X_val_text)

X_test_tfidf = tfidf.transform(X_test_text)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

In [ ]:
tfidf_lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

tfidf_lr.fit(
    X_train_tfidf,
    y_train
)

In [ ]:
val_prob = tfidf_lr.predict_proba(X_val_tfidf)[:, 1]

print(val_prob[:10])

In [ ]:
val_pr_auc = average_precision_score(
    y_val,
    val_prob
)

print("Validation PR-AUC:", round(val_pr_auc, 4))

In [ ]:
precision, recall, thresholds = precision_recall_curve(
    y_val,
    val_prob
)

plt.figure(figsize=(8, 6))

plt.plot(
    recall,
    precision,
    label=f"TF-IDF + LR (AP = {val_pr_auc:.3f})"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — Validation Set")
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
thresholds_to_test = np.arange(
    0.05,
    0.96,
    0.01
)

threshold_results = []

for threshold in thresholds_to_test:

    val_pred = (
        val_prob >= threshold
    ).astype(int)

    precision_val = precision_score(
        y_val,
        val_pred,
        zero_division=0
    )

    recall_val = recall_score(
        y_val,
        val_pred,
        zero_division=0
    )

    f1_val = f1_score(
        y_val,
        val_pred,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_val,
        "recall": recall_val,
        "f1": f1_val
    })

threshold_results_df = pd.DataFrame(
    threshold_results
)

threshold_results_df.sort_values(
    "f1",
    ascending=False
).head(10)

In [ ]:
best_row = threshold_results_df.loc[
    threshold_results_df["f1"].idxmax()
]

best_threshold = best_row["threshold"]

print("Best threshold:", round(best_threshold, 2))
print("Validation Precision:", round(best_row["precision"], 4))
print("Validation Recall:", round(best_row["recall"], 4))
print("Validation F1:", round(best_row["f1"], 4))

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    threshold_results_df["threshold"],
    threshold_results_df["f1"],
    label="F1"
)

plt.axvline(
    best_threshold,
    linestyle="--",
    label=f"Best threshold = {best_threshold:.2f}"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Fraud F1")
plt.title("Threshold Selection Using Validation Set")
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    threshold_results_df["threshold"],
    threshold_results_df["precision"],
    label="Precision"
)

plt.plot(
    threshold_results_df["threshold"],
    threshold_results_df["recall"],
    label="Recall"
)

plt.axvline(
    best_threshold,
    linestyle="--",
    label=f"Selected threshold = {best_threshold:.2f}"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Precision-Recall Trade-off vs Threshold")
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
test_prob = tfidf_lr.predict_proba(
    X_test_tfidf
)[:, 1]
test_pred_tuned = (
    test_prob >= best_threshold
).astype(int)

In [ ]:
test_accuracy = accuracy_score(
    y_test,
    test_pred_tuned
)

test_precision = precision_score(
    y_test,
    test_pred_tuned,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_pred_tuned,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_pred_tuned,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test,
    test_prob
)

test_pr_auc = average_precision_score(
    y_test,
    test_prob
)

print("FINAL TEST PERFORMANCE")
print("=" * 45)

print(f"Threshold:      {best_threshold:.3f}")
print(f"Accuracy:       {test_accuracy:.4f}")
print(f"Fraud Precision:{test_precision:.4f}")
print(f"Fraud Recall:   {test_recall:.4f}")
print(f"Fraud F1:       {test_f1:.4f}")
print(f"ROC-AUC:        {test_roc_auc:.4f}")
print(f"PR-AUC:         {test_pr_auc:.4f}")

In [ ]:
test_pred_default = (
    test_prob >= 0.50
).astype(int)

default_f1 = f1_score(
    y_test,
    test_pred_default,
    zero_division=0
)

default_precision = precision_score(
    y_test,
    test_pred_default,
    zero_division=0
)

default_recall = recall_score(
    y_test,
    test_pred_default,
    zero_division=0
)

tuned_f1 = f1_score(
    y_test,
    test_pred_tuned,
    zero_division=0
)

tuned_precision = precision_score(
    y_test,
    test_pred_tuned,
    zero_division=0
)

tuned_recall = recall_score(
    y_test,
    test_pred_tuned,
    zero_division=0
)

comparison = pd.DataFrame({
    "Threshold": [0.50, best_threshold],
    "Precision": [default_precision, tuned_precision],
    "Recall": [default_recall, tuned_recall],
    "F1": [default_f1, tuned_f1]
})

comparison

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(
    y_test,
    test_pred_tuned
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Real", "Fraud"]
)

disp.plot()

plt.title(
    f"Confusion Matrix — TF-IDF + LR\n"
    f"Threshold = {best_threshold:.2f}"
)

plt.show()

In [ ]:
print(
    classification_report(
        y_test,
        test_pred_tuned,
        target_names=["Real", "Fraud"],
        digits=4
    )
)

### Threshold Selection Conclusion

The validation set was used to evaluate different classification thresholds.

Although a threshold of 0.68 provided higher fraud precision, it substantially reduced fraud recall on the untouched test set and resulted in a lower Fraud F1 score than the default 0.50 threshold.

Therefore, the default threshold of 0.50 is retained as the primary operating threshold for the final baseline model.

Threshold tuning is retained as an analysis of the precision-recall trade-off rather than as a performance improvement.

In [ ]:
# ============================================================
# STRUCTURED + TEXT FEATURE FUSION
# ============================================================

from scipy.sparse import hstack

X_struct_train = df.loc[
    train_idx,
    STRUCTURED_FEATURES
]

X_struct_val = df.loc[
    val_idx,
    STRUCTURED_FEATURES
]

X_struct_test = df.loc[
    test_idx,
    STRUCTURED_FEATURES
]
